In [8]:
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

from restaurant_visitor_eda.features import add_visitor_aggregations, build_features_advanced

In [9]:
from restaurant_visitor_eda.config import PROCESSED_DATA_DIR

df_base = pd.read_csv(PROCESSED_DATA_DIR / "train_baseline.csv", parse_dates=["visit_date"])

df_base = build_features_advanced(df_base)
df_base = df_base.sort_values("visit_date").reset_index(drop=True)

In [10]:
categorical_features = [
    "air_store_id",
    "air_genre_name",
    "day_of_week",
    "month",
    "day_pattern",
    "prefecture",
    "district",
    "block",
]
numeric_features = [
    "latitude",
    "longitude",
    "doy_sin",
    "doy_cos",
    "dow_sin",
    "dow_cos",
    "median_visitors_dow",
    "mean_visitors_dow",
    "min_visitors_dow",
    "max_visitors_dow",
    "median_visitors_total",
    "gw_genre_geo_median",
    "median_reserve_visitors_dow",
    "walk_in_ratio",
]
binary_features = ["is_gw", "has_gw_history"]

features = categorical_features + numeric_features + binary_features

tscv = TimeSeriesSplit(n_splits=15)

cv_scores = []

models = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(df_base)):
    print(f"\n--- Fold {fold + 1} ---")
    df_train_local = df_base.iloc[train_idx].copy()
    df_val_local = df_base.iloc[val_idx].copy()

    df_train_feat = add_visitor_aggregations(df_train_local, df_train=df_train_local)
    df_val_feat = add_visitor_aggregations(df_val_local, df_train=df_train_local)

    df_train_feat[categorical_features] = (
        df_train_feat[categorical_features].fillna("Unknown").astype(str)
    )
    df_val_feat[categorical_features] = (
        df_val_feat[categorical_features].fillna("Unknown").astype(str)
    )

    X_train = df_train_feat[features]
    y_train = np.log1p(df_train_feat["visitors"])
    X_val = df_val_feat[features]
    y_val = np.log1p(df_val_feat["visitors"])

    model = CatBoostRegressor(
        iterations=1000,
        learning_rate=0.05,
        depth=8,
        loss_function="RMSE",
        eval_metric="RMSE",
        cat_features=categorical_features,
        random_seed=42,
        od_type="Iter",
        od_wait=50,
    )

    model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True, verbose=0)
    models.append(model)

    best_score = model.get_best_score()["validation"]["RMSE"]
    print(f"Fold {fold + 1} RMSLE: {best_score:.4f}")
    cv_scores.append(best_score)

print(f"\n[TimeSeriesSplit] Mean RMSLE: {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")


--- Fold 1 ---
Fold 1 RMSLE: 0.5823

--- Fold 2 ---
Fold 2 RMSLE: 0.5560

--- Fold 3 ---
Fold 3 RMSLE: 0.7820

--- Fold 4 ---
Fold 4 RMSLE: 0.5737

--- Fold 5 ---
Fold 5 RMSLE: 0.5460

--- Fold 6 ---
Fold 6 RMSLE: 0.5577

--- Fold 7 ---
Fold 7 RMSLE: 0.5395

--- Fold 8 ---
Fold 8 RMSLE: 0.5606

--- Fold 9 ---
Fold 9 RMSLE: 0.5355

--- Fold 10 ---
Fold 10 RMSLE: 0.5721

--- Fold 11 ---
Fold 11 RMSLE: 0.5828

--- Fold 12 ---
Fold 12 RMSLE: 0.5386

--- Fold 13 ---
Fold 13 RMSLE: 0.5296

--- Fold 14 ---
Fold 14 RMSLE: 0.5372

--- Fold 15 ---
Fold 15 RMSLE: 0.5423

[TimeSeriesSplit] Mean RMSLE: 0.5691 ± 0.0594


In [11]:
df_test_raw = pd.read_csv(PROCESSED_DATA_DIR / "test_features.csv")

df_test_raw[categorical_features] = df_test_raw[categorical_features].fillna("Unknown").astype(str)

X_test = X_test = df_test_raw[features]

test_preds_log = np.zeros(len(X_test))

for model in models:
    test_preds_log += model.predict(X_test)

test_preds_log /= len(models)

final_test_preds = np.expm1(test_preds_log)

df_test_raw["visit_date"] = pd.to_datetime(df_test_raw["visit_date"])

submission = pd.DataFrame(
    {
        "id": df_test_raw["air_store_id"]
        + "_"
        + df_test_raw["visit_date"].dt.strftime("%Y-%m-%d"),
        "visitors": final_test_preds,
    }
)

submission_path = "submission_catboost_tss.csv"
submission.to_csv(submission_path, index=False)

submission.head()

,id,visitors
0,air_00a91d42b08b08d9_2017-04-23,2.965284
1,air_08cb3c4ee6cd6a22_2017-04-23,16.664008
2,air_f8233ad00755c35c_2017-04-23,7.547067
3,air_234d3dbf7f3d5a50_2017-04-23,9.087993
4,air_a563896da3777078_2017-04-23,32.175006
